# Velocity Validator Introduction

## 1. Introduction

### 1.1 General Overview

This notebook introduces the velocity-field validator developed for active Beris-Edwards systems. The goal of the validator is to take processed simulation outputs, reconstruct the relevant tensor and flow quantities, and then test whether the simulated data are consistent with the target continuum equation.

In practice, the validator is designed to answer questions such as:

- whether the simulated velocity dynamics match the intended active Beris-Edwards momentum equation,
- whether the recovered coefficients agree with the solver input parameters,

In principle, the same framework could also be used to test whether experimental data are consistent with this equation and to estimate the corresponding parameters. However, that use case would first require a separate denoising step. The present validator does not include such a denoising pipeline. Because its primary design goal is solver validation, it assumes that the input data are already sufficiently smooth.

The present notebook focuses on validating the velocity equation. Validation of the $Q$-tensor equation is introduced separately in the $Q$ validator tutorial.

### 1.2 Method: SINDy

The main identification tool used here is [SINDy](https://pysindy.readthedocs.io/en/latest/), short for Sparse Identification of Nonlinear Dynamics. The basic idea is to construct a library of candidate terms that may appear in the governing equation, evaluate those terms from data, and then use sparse regression to determine which terms are actually needed and what their coefficients are.

Using the PySINDy package is straightforward. It can be installed with:

```bash
pip install pysindy
```

To understand the basic spirit of this validator, we could consider the simple diffusion equation $\partial_t\phi=D\nabla^2\phi$. Starting from numerical snapshots of $\phi$, we first sample a set of spatial points and time levels. At each sampled point, finite differences are used to calculate $\partial_t\phi$ and $\nabla^2\phi$ from the numerical solution. These values form a linear regression problem, $y_i=D x_i$, where $y_i=(\partial_t\phi)_i$ and $x_i=(\nabla^2\phi)_i$. Fitting this relation over all sampled points gives the diffusion coefficient $D$. The velocity validator follows the same general idea, except that the regression contains more candidate terms and all three components of the velocity field are included together in the regression.

Because of the special structure of the velocity equation, the SINDy regression used here requires some additional operations. The reason for these operations will become clear from the target equation introduced in Section 1.3, and their implementation will be described later in this tutorial.

This approach converts equation validation into a concrete regression problem with measurable outputs such as recovered coefficients, $R^2$, and relative residual.

### 1.3 Target Equation

The velocity validator targets an incompressible active Beris-Edwards momentum equation with the incompressibility condition

$$
\nabla\cdot\mathbf{u}=0.
$$

The equation is normalized by the viscous term, so its left-hand side is $\nabla^2\mathbf{u}$. The right-hand side consists of the following terms:

| Meaning | Formula | Coefficient |
| --- | --- | --- |
| time derivative | $\partial_t\mathbf{u}$ | $1/\nu$ |
| material advection | $(\mathbf{u}\cdot\nabla)\mathbf{u}$ | $\lambda_{\mathrm{adv}}/\nu$ |
| pressure gradient | $\nabla p/\rho$ | $1/\nu$ |
| active force | $\nabla\cdot Q$ | $-\alpha/\nu$ |
| passive backflow | $-\nabla\cdot\sigma^{\mathrm{passive}}$ | $-c_{\mathrm{bf}}/\nu$ |
| linear velocity damping | $\mathbf{u}$ | $-c_{\mathrm{lin}}/\nu$ |
| quadratic velocity diagnostic | $|\mathbf{u}|\mathbf{u}$ | $-c_{\mathrm{quad}}/\nu$ (expected $0$) |

The passive backflow is the force generated by the divergence of the passive nematic stress, where the passive stress used by the validator is

$$
\begin{aligned}
\sigma^{\mathrm{passive}}={}&-fI
-2\lambda_2\left(Q+\frac{I}{3}\right)(Q:H)\\
&+\lambda_2\left[H\left(Q+\frac{I}{3}\right)
+\left(Q+\frac{I}{3}\right)H\right]\\
&+HQ-QH+L_1(\nabla Q\odot\nabla Q).
\end{aligned}
$$

Here $A:B=\operatorname{tr}(A^TB)$, and $\nabla Q\odot\nabla Q$ denotes the matrix whose $(i,j)$ entry is $(\partial_iQ):(\partial_jQ)$. The flow-alignment parameter is $\lambda_2$, the coefficient of $EQ+QE$ in the $Q$ equation. It is not $\lambda_1$ or $\lambda_3$ because, in the single-parameter Beris--Edwards flow-alignment structure, these are the dependent coefficients of the accompanying terms $E$ and $(Q:E)Q$, respectively: $\lambda_1=2\lambda_2/3$ and $\lambda_3=-2\lambda_2$. The molecular field $H$ and free-energy density $f$ are

$$
H=-a_2Q
-a_3\left[Q^2-\frac{1}{3}\operatorname{tr}(Q^2)I\right]
-a_4\operatorname{tr}(Q^2)Q
+L_1\nabla^2Q,
$$

$$
f=\frac{1}{2}a_2\operatorname{tr}(Q^2)
+\frac{1}{3}a_3\operatorname{tr}(Q^3)
+\frac{1}{4}a_4\left[\operatorname{tr}(Q^2)\right]^2
+\frac{1}{2}L_1|\nabla Q|^2.
$$

### 1.4 Special Treatment

Two aspects of velocity-equation validation require special treatment. The first is the pressure term.

#### 1.4.1 Pressure and the Weak Form

Unlike the other terms, the pressure gradient cannot generally be constructed from the stored velocity and $Q$ fields alone. To eliminate it without reconstructing the pressure, the validator divides the spatial domain into local patches $\mathcal{D}_k$ and constructs compactly supported divergence-free test fields $\mathbf{w}$ on each patch:

$$
\nabla\cdot\mathbf{w}=0,
\qquad
\mathbf{w}=0 \quad\text{on }\partial\mathcal{D}_k.
$$

The momentum equation is multiplied by $\mathbf{w}$ and integrated over the patch. Writing the reduced pressure as $\pi=p/\rho$, integration by parts transforms its contribution into

$$
-\int_{\mathcal{D}_k}\mathbf{w}\cdot\nabla\pi\,dV
=\int_{\mathcal{D}_k}\pi\,\nabla\cdot\mathbf{w}\,dV
-\int_{\partial\mathcal{D}_k}\pi\,\mathbf{w}\cdot\mathbf{n}\,dS
=0.
$$

The volume integral vanishes because $\nabla\cdot\mathbf{w}=0$, while the boundary integral vanishes because $\mathbf{w}$ has compact support inside the patch. Therefore, the pressure is removed exactly from the weak equation rather than estimated or included as a regression feature.

For the remaining candidate terms $\Theta_j$, the equation on each patch becomes

$$
\int_{\mathcal{D}_k}\mathbf{w}\cdot\partial_t\mathbf{u}\,dV
=\sum_j c_j\int_{\mathcal{D}_k}\mathbf{w}\cdot\Theta_j\,dV.
$$

Each choice of patch, test field, and valid time level provides one scalar regression sample. Collecting these weak-form samples produces the linear system used by SINDy to recover the coefficients $c_j$.

In addition, the local spatial integration averages the fields within each patch and therefore also provides a smoothing effect. The patch size is consequently a key validator parameter and is set through the input parameter `v_patch_half_widths`. Larger patches provide stronger spatial smoothing, while smaller patches retain more local spatial detail.

#### 1.4.2 Passive Backflow

If the passive backflow is decomposed into all of its constituent structures and fitted directly, the velocity regression must recover almost the entire set of $Q$-equation parameters, including $a_2$, $a_3$, $a_4$, $L_1$, and $\lambda_2$. This produces a more complicated regression problem and may introduce stronger collinearity among the candidate terms.

The validator therefore takes a different approach. These parameters are supplied as inputs and used to construct the complete passive-backflow term $-\nabla\cdot\sigma^{\mathrm{passive}}$. The velocity regression then fits only its overall coefficient $c_{\mathrm{bf}}$. The required parameter values can first be obtained by applying the $Q$-equation validator to the same data. If the solver includes the complete passive backflow without any additional scaling, the expected result is $c_{\mathrm{bf}}=1$.

#### 1.4.3 Viscosity Normalization

Active-liquid-crystal flows are typically studied in the low-Reynolds-number or even zero-Reynolds-number limit. In this regime, $\partial_t\mathbf{u}$ can be very small. Using it as the normalized left-hand-side target would therefore amplify the numerical noise in the time derivative and make the regression unstable. The validator instead places the viscous term $\nabla^2\mathbf{u}$ on the left-hand side and normalizes its coefficient to $1$.

This choice has a cost. If the physical equation contains viscosity $\nu$ and coefficients $c_j$, the viscosity-normalized regression recovers combinations proportional to $c_j/\nu$, rather than the absolute coefficients $c_j$. The absolute coefficient scale therefore cannot be determined from this normalized regression alone. To restore that scale, the validator includes a manually supplied `viscosity` parameter. Its default value is `1.0`; when the physical viscosity is known, the supplied value is used to rescale all reported coefficients back to the standard form with $\partial_t\mathbf{u}$ on the left.


## 2. Usage

### 2.1 Generate and Prepare the Simulation Data

Run the solver once and save both the $Q$ field and the velocity field at every time step. The number of saved time steps can be chosen according to the needs of the validation. When possible, use a sufficiently complex state as the initial condition so that the candidate terms exhibit distinct spatial structures and are less likely to be strongly collinear.

After the simulation, convert the numerical solutions into two NumPy files with the following names and layouts:

| File name | Array shape | Component order |
| --- | --- | --- |
| `q.npy` | $(T,N_x,N_y,N_z,5)$ | $(Q_{xx},Q_{xy},Q_{xz},Q_{yy},Q_{yz})$ |
| `velocity.npy` | $(T,N_x,N_y,N_z,3)$ | $(u_x,u_y,u_z)$ |

Here, $T$ is the number of saved time levels. The fields must be stored in chronological order, and the two files must share the same $(T,N_x,N_y,N_z)$ layout. Because $Q$ is symmetric and traceless, the validator reconstructs the remaining tensor components from the five stored components, including $Q_{zz}=-(Q_{xx}+Q_{yy})$.


### 2.2 Run the Validator and Set the Parameters

Pass the two prepared files to `discover_from_processed_npy_local_weak_form(...)`. Set `dt` to the time interval between two saved fields and `spacings` to the grid spacings $(\Delta x,\Delta y,\Delta z)$. To include passive backflow, supply $a_2$, $a_3$, $a_4$, $L_1$, and $\lambda_2$. These values can be obtained directly by first applying `discover_q_from_processed_npy(...)` to the same data. Other parameters use their default values in this minimal example.


In [1]:
from pathlib import Path
import sys

# Locate the example directory whether Jupyter starts here or at the repository root.
example_dir = Path.cwd()
if example_dir.name != "example":
    example_dir = example_dir / "example"

validator_root = example_dir.parent
checkq_root = validator_root.parent / "CheckQ_SINDy"
checkq_src = checkq_root / "src"
# Make the packaged validators and their source library importable.
for import_path in (checkq_root, checkq_src):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from app import (
    discover_from_processed_npy_local_weak_form,
    discover_q_from_processed_npy,
)

# First recover the Q-equation parameters needed to construct passive backflow.
# q_result = discover_q_from_processed_npy(
#     q_npy_path="q.npy",
#     velocity_npy_path="velocity.npy",
#     dt=1.0,
#     spacings=(1.0, 1.0, 1.0),
#     gamma=1.0,
# )
# q_coefficients = q_result["coefficients"]
# velocity_result = discover_from_processed_npy_local_weak_form(
#     q_npy_path="q.npy",
#     velocity_npy_path="velocity.npy",
#     dt=1.0,
#     spacings=(1.0, 1.0, 1.0),
#     is_v_include_passive_backflow=True,
#     v_a2=q_coefficients["a2"],
#     v_a3=q_coefficients["a3"],
#     v_a4=q_coefficients["a4"],
#     v_kappa=q_coefficients["L1"],
#     v_flow_alignment_xi=q_coefficients["lambda_2"],
# )


### 2.3 Ludwig Example

[Ludwig](https://ludwig.epcc.ed.ac.uk/) is an open-source parallel lattice-Boltzmann code developed largely at the University of Edinburgh for simulations of complex fluids, including liquid crystals.

Here, the packaged validators are applied to $Q$ and velocity fields generated by Ludwig. The $Q$ validator is run first to recover the parameters required by the passive stress. Those recovered values are then passed directly to the velocity validator. The velocity fit uses patch half-widths $(8,8,8)$, one test field per patch, and viscosity normalization. Because Ludwig is an independent solver, recovering its intended velocity equation also provides a separate check of the weak-form velocity validator itself.


In [2]:
ludwig_data_dir = example_dir / "data" / "ludwig_checkpoint_dense"

# Recover the Q-equation parameters from the same Ludwig data.
ludwig_q_result = discover_q_from_processed_npy(
    q_npy_path=ludwig_data_dir / "cache_q_stack.npy",
    velocity_npy_path=ludwig_data_dir / "cache_u_stack.npy",
    dt=1.0,
    spacings=(1.0, 1.0, 1.0),
    gamma=2.0,
)
ludwig_q_coefficients = ludwig_q_result["coefficients"]

# Construct the complete passive backflow and fit the velocity equation.
ludwig_velocity_result = discover_from_processed_npy_local_weak_form(
    q_npy_path=ludwig_data_dir / "cache_q_stack.npy",
    velocity_npy_path=ludwig_data_dir / "cache_u_stack.npy",
    dt=1.0,
    spacings=(1.0, 1.0, 1.0),
    viscosity=0.5,
    v_patch_half_widths=(8, 8, 8),
    v_patch_strides=(16, 16, 16),
    num_test_fields_per_patch=1,
    is_v_include_passive_backflow=True,
    v_a2=ludwig_q_coefficients["a2"],
    v_a3=ludwig_q_coefficients["a3"],
    v_a4=ludwig_q_coefficients["a4"],
    v_kappa=ludwig_q_coefficients["L1"],
    v_flow_alignment_xi=ludwig_q_coefficients["lambda_2"],
    is_verbose=True,
)


Starting local weak-form fit: velocity shape=(21, 128, 128, 128, 3), Q shape=(21, 128, 128, 128, 3, 3), normalization_mode=viscosity, spatial_derivative_method=finite_difference, time_derivative_method=central, num_test_fields_per_patch=1
Computing dt(v), v, |v|v, (v·grad)v, Laplacian(v), and div(Q), and passive backflow patch by patch...
Starting local patch integration: 343 patches, half_widths=(8, 8, 8), strides=(16, 16, 16)
Completed patch 1/343 (elapsed=1.1s, eta=375.6s)
Completed patch 10/343 (elapsed=11.2s, eta=372.0s)
Completed patch 20/343 (elapsed=22.2s, eta=358.6s)
Completed patch 30/343 (elapsed=33.6s, eta=351.1s)
Completed patch 40/343 (elapsed=44.6s, eta=337.5s)
Completed patch 50/343 (elapsed=55.2s, eta=323.3s)
Completed patch 60/343 (elapsed=65.8s, eta=310.1s)
Completed patch 70/343 (elapsed=76.6s, eta=298.6s)
Completed patch 80/343 (elapsed=87.1s, eta=286.5s)
Completed patch 90/343 (elapsed=98.0s, eta=275.5s)
Completed patch 100/343 (elapsed=109.6s, eta=266.4s)
Complet

In [3]:
ludwig_velocity_result

{'coefficients': {'dt': 0.904978967824952,
  'material_advection': -1.0877959775475272,
  'viscosity': 0.5,
  'active_force': -0.0019965401871236575,
  'passive_backflow': 0.9412439680964426,
  'linear_velocity': 0.00041347157428838106,
  'quadratic_velocity': -0.009275344551682623},
 'input_parameters': {'q_npy_path': 'D:\\Document\\GitHub\\ActiveBE_Validator\\example\\data\\ludwig_checkpoint_dense\\cache_q_stack.npy',
  'velocity_npy_path': 'D:\\Document\\GitHub\\ActiveBE_Validator\\example\\data\\ludwig_checkpoint_dense\\cache_u_stack.npy',
  'dt': 1.0,
  'spacings': (1.0, 1.0, 1.0),
  'viscosity': 0.5,
  'v_patch_half_widths': (8, 8, 8),
  'v_patch_strides': (16, 16, 16),
  'v_time_derivative_method': 'central',
  'num_test_fields_per_patch': 1,
  'is_v_include_passive_backflow': True,
  'v_a2': -0.0003323436830255028,
  'v_a3': -0.030859452602501797,
  'v_a4': 0.030868289403489308,
  'v_kappa': 0.004112658301150009,
  'v_flow_alignment_xi': 1.0045474006248962,
  'v_threshold': 1e-

### 2.4 Understanding the Results

The returned `ProcessedOutputWeakFormDiscoveryResult` contains a `velocity_result` field holding the complete local weak-form fit. Its main outputs are:

| Output | Meaning |
| --- | --- |
| `dt_coefficient` | Coefficient of $\partial_t\mathbf{u}$ after multiplying the viscosity-normalized result by the input viscosity. It is expected to be close to $1$ when the supplied viscosity is correct. |
| `material_coefficient` | Reported physical coefficient of $(\mathbf{u}\cdot\nabla)\mathbf{u}$ after viscosity rescaling. |
| `viscosity_coefficient` | Input physical viscosity used to restore the coefficient scale. |
| `q_divergence_coefficient` | Reported physical active-force coefficient multiplying $\nabla\cdot Q$. |
| `passive_backflow_coefficient` | Reported physical coefficient $c_{\mathrm{bf}}$ multiplying the complete passive-backflow term. |
| `linear_velocity_coefficient` | Reported physical coefficient of the linear velocity term $\mathbf{u}$. |
| `quadratic_velocity_coefficient` | Reported diagnostic coefficient of $|\mathbf{u}|\mathbf{u}$; it is expected to be zero when this term is absent from the target equation. |
| `sample_count` | Number of scalar weak-form samples used in the regression. |

Because the validator uses viscosity normalization, the raw fitted equation has the form $\nabla^2\mathbf{u}=c_t^{(\mathrm{visc})}\partial_t\mathbf{u}+\sum_j c_j^{(\mathrm{visc})}\Theta_j$. Given the input viscosity $\nu$, the reported time-derivative coefficient is $\nu c_t^{(\mathrm{visc})}$, the viscosity coefficient is $\nu$, and every reported right-hand-side coefficient is $-\nu c_j^{(\mathrm{visc})}$. Thus the reported coefficients have the scale and signs of the standard equation with $\partial_t\mathbf{u}$ on the left.

Let $y_i$ denote the scalar weak-form targets, let $\hat{y}_i$ denote the corresponding predictions, and define $r_i=y_i-\hat{y}_i$. For a total of $N$ samples, the fit metrics are

| Metric | Calculation | Meaning |
| --- | --- | --- |
| `r2` | $R^2=1-\dfrac{\sum_{i=1}^{N}r_i^2}{\sum_{i=1}^{N}(y_i-\bar{y})^2}$ | Fraction of the weak-form target variation explained by the fitted equation. A value close to $1$ indicates a strong fit. |
| `relative_residual` | $\dfrac{\lVert y-\hat{y}\rVert_2}{\lVert y\rVert_2}$ | Residual magnitude relative to the target magnitude. A value of $0$ represents exact recovery. |
| `rmse` | $\sqrt{\dfrac{1}{N}\sum_{i=1}^{N}r_i^2}$ | Root-mean-square error per weak-form sample, with the same units as the selected normalized target. |

The input parameters used to generate these results are described in the next section.


### 2.5 Input Parameters

The input parameters of `discover_from_processed_npy_local_weak_form(...)` are:

| Parameter | Default | Meaning |
| --- | --- | --- |
| `q_npy_path` | Required | Path to the NumPy file containing the five independent components of $Q$ with shape $(T,N_x,N_y,N_z,5)$ and component order $(Q_{xx},Q_{xy},Q_{xz},Q_{yy},Q_{yz})$. |
| `velocity_npy_path` | Required | Path to the NumPy file containing the velocity field with shape $(T,N_x,N_y,N_z,3)$ and component order $(u_x,u_y,u_z)$. |
| `dt` | `1.0` | Time interval between two consecutively stored fields. |
| `spacings` | `(1.0, 1.0, 1.0)` | Spatial grid spacings $(\Delta x,\Delta y,\Delta z)$. |
| `viscosity` | `1.0` | Physical kinematic viscosity $\nu$ used to rescale the viscosity-normalized regression coefficients for reporting. The reported time-derivative coefficient is multiplied by $\nu$, while every other fitted right-hand-side coefficient is multiplied by $-\nu$. |
| `v_patch_half_widths` | `(4, 4, 4)` | Patch half-widths in grid points along the three spatial directions. A patch centered at a given point extends by the specified number of grid points on each side. Larger patches provide stronger spatial averaging. |
| `v_patch_strides` | `(16, 16, 16)` | Distances in grid points between neighboring patch centers. Smaller strides produce more overlapping patches and more regression samples. |
| `v_time_derivative_method` | `"central"` | Finite-difference method for $\partial_t\mathbf{u}$. `"central"` uses centered time differences, while `"forward"` can more closely match a forward time-marching update. |
| `num_test_fields_per_patch` | `1` | Number of deterministic compactly supported divergence-free test fields constructed on each patch. Supported values are `1`, `3`, `12`, and `30`. The same inputs always generate the same fields; no random seed is used. Larger values provide more weak-form regression samples per patch. |
| `is_v_include_passive_backflow` | `False` | Whether to include $-\nabla\cdot\sigma^{\mathrm{passive}}$ in the candidate library. If enabled, the material parameters required to construct the passive stress must be supplied through `v_a2`, `v_a3`, `v_a4`, and `v_kappa`. |
| `v_a2` | `None` | Quadratic bulk free-energy coefficient $a_2$ used to construct the molecular field and passive stress. Required when passive backflow is included. |
| `v_a3` | `None` | Cubic bulk free-energy coefficient $a_3$ used to construct the molecular field and passive stress. Required when passive backflow is included. |
| `v_a4` | `None` | Quartic bulk free-energy coefficient $a_4$ used to construct the molecular field and passive stress. Required when passive backflow is included. |
| `v_kappa` | `None` | One-constant elastic coefficient used in the molecular field and passive stress; it corresponds to $L_1$ in this tutorial. Required when passive backflow is included. |
| `v_flow_alignment_xi` | `1.0` | Flow-alignment coefficient used in the passive stress. In the notation of this tutorial, this is $\lambda_2$. |
| `v_threshold` | `1e-10` | Sparsity threshold used by sequentially thresholded least squares. Candidate terms below the threshold are removed from the fitted model. |
| `v_optimizer_alpha` | `0.0` | Ridge regularization strength used by the SINDy optimizer. This numerical parameter is unrelated to the physical active-force coefficient $\alpha$. |
| `is_v_normalize_columns` | `False` | Whether to normalize candidate-library columns before sparse regression. |
| `is_verbose` | `False` | Whether to print progress information while weak-form samples are constructed and fitted. |
| `progress_interval` | `10` | Number of completed patches between progress messages when `is_verbose=True`. |
